In [8]:
from data_loader import load_mimic_tables                                      
                                                                                 
tables = load_mimic_tables()                                                                                                        
                                                                                 
# See all available tables:                                                    
print(tables.keys())  

/Users/huseyin/Codes/gpt-medic/explore/data_loader.py:26: DtypeWarning: Columns (0: administration_type, 1: barcode_type, 2: reason_for_no_barcode, 3: complete_dose_not_given, 4: dose_due, 5: dose_due_unit, 6: dose_given, 7: dose_given_unit, 8: will_remainder_of_dose_be_given, 9: product_unit, 10: product_code, 11: product_description, 12: product_description_other, 13: infusion_rate_adjustment, 14: infusion_rate_unit, 15: route, 16: infusion_complete, 17: completion_interval, 18: new_iv_bag_hung, 19: continued_infusion_in_other_location, 20: restart_interval, 21: side, 22: site, 23: non_formulary_visual_verification) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[table_name] = pd.read_csv(file, compression="gzip")


dict_keys(['hosp.poe', 'hosp.d_hcpcs', 'hosp.poe_detail', 'hosp.patients', 'hosp.diagnoses_icd', 'hosp.emar_detail', 'hosp.provider', 'hosp.prescriptions', 'hosp.drgcodes', 'hosp.d_icd_diagnoses', 'hosp.d_labitems', 'hosp.transfers', 'hosp.admissions', 'hosp.labevents', 'hosp.pharmacy', 'hosp.procedures_icd', 'hosp.hcpcsevents', 'hosp.services', 'hosp.d_icd_procedures', 'hosp.omr', 'hosp.emar', 'hosp.microbiologyevents', 'icu.datetimeevents', 'icu.caregiver', 'icu.ingredientevents', 'icu.inputevents', 'icu.procedureevents', 'icu.d_items', 'icu.chartevents', 'icu.icustays', 'icu.outputevents'])


In [22]:
tables["hosp.procedures_icd"]

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
0,10011398,27505812,3,2146-12-15,3961,9
1,10011398,27505812,2,2146-12-15,3615,9
2,10011398,27505812,1,2146-12-15,3614,9
3,10014729,23300884,4,2125-03-23,3897,9
4,10014729,23300884,1,2125-03-20,3403,9
...,...,...,...,...,...,...
717,10004733,27411876,3,2174-12-20,4513,9
718,10021118,24490144,4,2161-11-19,5A1221Z,10
719,10021118,24490144,3,2161-11-19,06BP4ZZ,10
720,10021118,24490144,1,2161-11-19,02100Z9,10


In [9]:
from tokenizer.flextpp import PatientTimelineTokenizer, Event

# Create and fit the tokenizer
tokenizer = PatientTimelineTokenizer()
tokenizer.fit(tables)

print("Tokenizer fitted successfully!")
print(f"Combined vocabulary size: {len(tokenizer.get_combined_vocabulary())}")

Tokenizer fitted successfully!
Combined vocabulary size: 3019


In [10]:
# Get one example hospital admission
admissions = tables['hosp.admissions']
example_hadm_id = admissions['hadm_id'].iloc[0]

print(f"Example hadm_id: {example_hadm_id}")

# Tokenize the session
result = tokenizer.tokenize_session(example_hadm_id, tables)

print(f"\nPatient: {result['subject_id']}")
print(f"Admission: {result['admit_time']}")
print(f"Discharge: {result['discharge_time']}")
print(f"Total events: {len(result['events'])}")

Example hadm_id: 24181354

Patient: 10004235
Admission: 2196-02-24 14:38:00
Discharge: 2196-03-04 14:02:00
Total events: 875


In [11]:
# Show first 20 events
print("First 20 events:\n")
print(f"{'Time (days)':<12} {'Type':<6} {'Tokens'}")
print("-" * 80)

for event in result['events'][:20]:
    print(f"{event.time:<12.4f} {event.event_type:<6} {event.tokens}")

First 20 events:

Time (days)  Type   Tokens
--------------------------------------------------------------------------------
-0.6097      PROC   <PROC> <3891>
-0.6097      PROC   <PROC> <9671>
0.0000       DEMO   <SEX> <M> <MARITAL> <SINGLE> <RACE> <BLACK_CAPE_VERDEAN>
0.0000       DIAG   <ICD> <03842>
0.0000       DIAG   <ICD> <78551>
0.0000       DIAG   <ICD> <5845>
0.0000       DIAG   <ICD> <570>
0.0000       DIAG   <ICD> <51881>
0.0000       DIAG   <ICD> <486>
0.0000       DIAG   <ICD> <34830>
0.0000       DIAG   <ICD> <5761>
0.0000       DIAG   <ICD> <42821>
0.0000       DIAG   <ICD> <29900>
0.0000       DIAG   <ICD> <2762>
0.0000       DIAG   <ICD> <75169>
0.0000       DIAG   <ICD> <4254>
0.0000       DIAG   <ICD> <99592>
0.0000       DIAG   <ICD> <42731>
0.0000       DIAG   <ICD> <7906>
0.0000       DIAG   <ICD> <28749>


In [12]:
# Show one event in detail
print("Example Event structure:\n")
event = result['events'][5] if len(result['events']) > 5 else result['events'][0]
print(f"Event(")
print(f"    time={event.time},")
print(f"    tokens='{event.tokens}',")
print(f"    event_type='{event.event_type}',")
print(f"    raw_data={event.raw_data}")
print(f")")

Example Event structure:

Event(
    time=0.0,
    tokens='<ICD> <5845>',
    event_type='DIAG',
    raw_data={'icd_code': '5845', 'seq_num': 3}
)


In [17]:
tables.keys()

dict_keys(['hosp.poe', 'hosp.d_hcpcs', 'hosp.poe_detail', 'hosp.patients', 'hosp.diagnoses_icd', 'hosp.emar_detail', 'hosp.provider', 'hosp.prescriptions', 'hosp.drgcodes', 'hosp.d_icd_diagnoses', 'hosp.d_labitems', 'hosp.transfers', 'hosp.admissions', 'hosp.labevents', 'hosp.pharmacy', 'hosp.procedures_icd', 'hosp.hcpcsevents', 'hosp.services', 'hosp.d_icd_procedures', 'hosp.omr', 'hosp.emar', 'hosp.microbiologyevents', 'icu.datetimeevents', 'icu.caregiver', 'icu.ingredientevents', 'icu.inputevents', 'icu.procedureevents', 'icu.d_items', 'icu.chartevents', 'icu.icustays', 'icu.outputevents'])

In [13]:
# Check the procedure dates
procedures = tables['hosp.procedures_icd']
session_procs = procedures[procedures['hadm_id'] == example_hadm_id].copy()

print(f"Admission time: {result['admit_time']}")
print(f"Discharge time: {result['discharge_time']}")
print(f"\nProcedure chartdates:")
print(session_procs[['icd_code', 'chartdate']].head(10))

Admission time: 2196-02-24 14:38:00
Discharge time: 2196-03-04 14:02:00

Procedure chartdates:
   icd_code   chartdate
42     3891  2196-02-24
43     5187  2196-02-26
44     9671  2196-02-24
45     3897  2196-02-25
46     3995  2196-02-26


In [14]:
# Show the raw data
print("Admission datetime:", result['admit_time'])
print()

# Show procedure raw data
procs = tables['hosp.procedures_icd']
session_procs = procs[procs['hadm_id'] == example_hadm_id][['icd_code', 'chartdate']]
print("Procedure data (only has DATE, no time):")
print(session_procs.head())

Admission datetime: 2196-02-24 14:38:00

Procedure data (only has DATE, no time):
   icd_code   chartdate
42     3891  2196-02-24
43     5187  2196-02-26
44     9671  2196-02-24
45     3897  2196-02-25
46     3995  2196-02-26


In [15]:
import pandas as pd

# When we convert date to timestamp, it becomes midnight (00:00:00)
proc_date = pd.to_datetime("2196-02-24")
admit_time = pd.to_datetime("2196-02-24 14:38:00")

print(f"Procedure timestamp (from date): {proc_date}")
print(f"Admission timestamp:             {admit_time}")
print()
print(f"Difference: {proc_date - admit_time}")
print(f"In days: {(proc_date - admit_time).total_seconds() / 86400:.4f}")

Procedure timestamp (from date): 2196-02-24 00:00:00
Admission timestamp:             2196-02-24 14:38:00

Difference: -1 days +09:22:00
In days: -0.6097


In [23]:
# Check ICD versions in procedures
procs = tables['hosp.procedures_icd']
print("Columns:", procs.columns.tolist())
print()
print("ICD version distribution:")
print(procs['icd_version'].value_counts())

Columns: ['subject_id', 'hadm_id', 'seq_num', 'chartdate', 'icd_code', 'icd_version']

ICD version distribution:
icd_version
9     401
10    321
Name: count, dtype: int64


In [24]:
# Show example codes from each version
print("ICD-9 procedure codes (sample):")
print(procs[procs['icd_version'] == 9]['icd_code'].head(10).tolist())
print()
print("ICD-10 procedure codes (sample):")
print(procs[procs['icd_version'] == 10]['icd_code'].head(10).tolist())

ICD-9 procedure codes (sample):
['3961', '3615', '3614', '3897', '3403', '3404', '3452', '3845', '0390', '0391']

ICD-10 procedure codes (sample):
['02H633Z', '0BH17EZ', 'B211YZZ', '3E0G76Z', '0BCH8ZZ', '02HV33Z', '3E0436Z', '0BCJ8ZZ', '03LP3DZ', 'B518YZA']
